# MSUN Baseline - Fair Reproduction for Comparison

**Purpose.** This notebook re-implements the method of Wu et al. (2023), *Plant Phenomics* 5:0038,
"From Laboratory to Field: Unsupervised Domain Adaptation for Plant Disease Recognition in the Wild"
(MSUN). It runs on your exact PlantVillage -> PlantDoc 3-class split. The goal is a comparison with
MicroBrainNet-DA v7 under an identical-data protocol.

**What is reproduced (from the paper text).**
- ResNet-50 backbone, ImageNet-pretrained.
- Multi-representation module: four Inception-style substructures on top of the early CNN layers.
  - Sub1: conv1x1, conv5x5, average pool.
  - Sub2: conv1x1, conv3x3, conv3x3, average pool.
  - Sub3: conv1x1, average pool.
  - Sub4: average pool, conv1x1, average pool.
- Subdomain adaptation: LMMD (local MMD) on the representation vector of every substructure.
  Source weights come from true labels. Target weights come from the softmax of the current prediction.
- Auxiliary uncertainty regularization: entropy minimization on target predictions.
- Loss: cross-entropy + lambda * sum of LMMD terms + gamma * entropy.
- lambda ramps from 0 to 1 with 2/(1+exp(-10p)) - 1. gamma = 0.1, the value the paper reports as best.
- SGD with momentum 0.9. The new layers get 10x the learning rate.
- Equal-sized source and target batches. Resize to 256, random crop 224, random flip.
- Non-adversarial. Target labels are never used in the training loss.

**What the paper does not specify (assumed here, all are config constants).**
- Where the early CNN `g` ends. Default: after ResNet-50 `layer3`.
- Channel width of each substructure. Default: 128.
- Base learning rate, weight decay and batch size.
- Kernel bandwidth details. The DSAN-style multi-Gaussian kernel is used.

No official code link is given in the paper. This is a reconstruction from the text.

**Protocol parity with your other baselines.**
- Same `LeafIndex` logic and same `stratified_split_indices(seed=42)`.
- Same 3 classes. Same split sizes: 2628/564/564 source and 364/79/79 target.
- Same "smoothed target-val accuracy" checkpoint rule (window 5, patience 15).
- Same bootstrap 95% CI. Same 224 px input as the EffNet baseline.
- Single seed (42) by your decision. The `SEEDS` list can be extended later.

Run this in the same Kaggle environment as the v7 notebook.

In [1]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, time, random, warnings, copy, math

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = "/kaggle/working/msun_baseline"
MODELS_DIR = f"{OUTPUT_DIR}/models"
TABLES_DIR = f"{OUTPUT_DIR}/tables"
for d in [MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- must match the v7 notebook exactly for an identical split ----
SOURCE_ALL = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
TARGET_ALL = "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset"

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]
NUM_CLASSES = len(CLASSES)

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---- input pipeline (paper: resize 256, random crop 224, random flip) ----
IMG_SIZE    = 224
RESIZE_SIZE = 256
BATCH_SIZE        = 16     # source batch (assumed; not stated in the paper)
TARGET_BATCH_SIZE = 16     # equal to the source batch, as the paper requires

# ---- protocol shared with the other baselines ----
EPOCHS        = 60         # paper uses 200; 60 + early stopping matches the EffNet baseline
PATIENCE      = 15
SMOOTH_WINDOW = 5
SELECTION     = "smoothed_target_val"   # "last" = strict UDA, no target labels for selection
SEEDS         = [42]       # single seed by decision; extend to [42, 1337, 2024] if time allows
PROBE_EPOCHS  = 2          # short run used only to read peak GPU memory

# ---- MSUN hyperparameters ----
G_UP_TO        = "layer3"  # ASSUMED: early CNN g(.) = ResNet-50 up to layer3 ("layer4" also allowed)
BRANCH_WIDTH   = 128       # ASSUMED: channels per substructure
LR_G           = 1e-3      # ASSUMED: base LR of the pretrained layers
LR_NEW_MULT    = 10        # paper: new layers get 10x the learning rate
MOMENTUM       = 0.9       # paper
WEIGHT_DECAY   = 5e-4      # ASSUMED
LAMBDA_MAX     = 1.0       # paper: lambda ramps 0 -> 1
LAMBDA_ALPHA   = 10.0      # paper: alpha = 10
GAMMA          = 0.1       # paper: best value in the sensitivity analysis

print(f"\nIMG={IMG_SIZE} (resize {RESIZE_SIZE}) | classes={CLASSES}")
print(f"epochs<={EPOCHS} patience={PATIENCE} selection={SELECTION} seeds={SEEDS}")
print(f"g up to {G_UP_TO} | branch width {BRANCH_WIDTH} | lr_g={LR_G} x{LR_NEW_MULT} new | "
      f"lambda<={LAMBDA_MAX} gamma={GAMMA}")

Device: cuda | PyTorch: 2.11.0+cu128
GPU: Tesla T4

IMG=224 (resize 256) | classes=['Tomato_Septoria_Leaf_Spot', 'Potato_Early_Blight', 'Corn_Leaf_Blight']
epochs<=60 patience=15 selection=smoothed_target_val seeds=[42]
g up to layer3 | branch width 128 | lr_g=0.001 x10 new | lambda<=1.0 gamma=0.1


## Part 1 - Data indexing and split (identical logic to v7)

In [2]:
# ============================================================
# PART 1: DATASET INDEX + SPLIT  (identical to v7 notebook)
# ============================================================

class LeafIndex:
    # Path/label index only. Walks root_dir recursively and pools every image under
    # any folder whose normalized name matches an alias for a class.

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)   # used UNLABELED in training (UDA)
tgt_val_idx   = tgt_index.subset(tgt_va_i)   # labeled, used only for checkpoint selection
tgt_test_idx  = tgt_index.subset(tgt_te_i)   # labeled, touched once at the very end

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Report the bootstrap CI.")

  PlantVillage color      : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
  PlantDoc                : OK  /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Report the bootstrap CI.


## Part 2 - Datasets and loaders

Training batches use the paper's augmentation (resize 256, random crop 224, flip). Evaluation uses
the same plain resize as the EffNet baseline, so all baselines are evaluated on identical pixels.
The target-train loader returns labels too, but the training loop never reads them.

In [3]:
# ============================================================
# PART 2: DATASETS AND LOADERS
# ============================================================

class LeafImageDataset(Dataset):
    def __init__(self, index_obj, transform):
        self.idx = index_obj
        self.transform = transform

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        img = Image.open(self.idx.img_paths[i]).convert("RGB")
        return self.transform(img), self.idx.labels[i]


train_tf = transforms.Compose([
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

src_train_ds = LeafImageDataset(src_train_idx, train_tf)
src_val_ds   = LeafImageDataset(src_val_idx,   eval_tf)
src_test_ds  = LeafImageDataset(src_test_idx,  eval_tf)
tgt_train_ds = LeafImageDataset(tgt_train_idx, train_tf)   # labels never used in training
tgt_val_ds   = LeafImageDataset(tgt_val_idx,   eval_tf)
tgt_test_ds  = LeafImageDataset(tgt_test_idx,  eval_tf)

def mk_loader(ds, bs, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=4,
                      pin_memory=True, drop_last=drop_last, persistent_workers=True)

# drop_last avoids a size-1 batch, which BatchNorm cannot handle in train mode
src_train_loader = mk_loader(src_train_ds, BATCH_SIZE, True, drop_last=True)
tgt_train_loader = mk_loader(tgt_train_ds, TARGET_BATCH_SIZE, True, drop_last=True)
src_val_loader   = mk_loader(src_val_ds,   BATCH_SIZE, False)
src_test_loader  = mk_loader(src_test_ds,  BATCH_SIZE, False)
tgt_val_loader   = mk_loader(tgt_val_ds,   BATCH_SIZE, False)
tgt_test_loader  = mk_loader(tgt_test_ds,  BATCH_SIZE, False)

def cycle(loader):
    while True:
        for batch in loader:
            yield batch

print(f"source train batches/epoch : {len(src_train_loader)}")
print(f"target train batches/epoch : {len(tgt_train_loader)} (cycled, unlabeled use)")
xb, yb = next(iter(src_train_loader))
print(f"batch shape: {tuple(xb.shape)}  label sample: {yb[:8].tolist()}")

source train batches/epoch : 164
target train batches/epoch : 22 (cycled, unlabeled use)
batch shape: (16, 3, 224, 224)  label sample: [1, 0, 1, 0, 0, 1, 2, 1]


## Part 3 - MSUN model

`g(.)` = ResNet-50 stem and stages up to `G_UP_TO`. `h(.)` = four substructures from the paper
(each ends with average pooling, then a global average pool gives one vector per substructure).
`c(.)` = one fully connected layer on the concatenated vectors. In training mode the model also
returns the four representation vectors, which the LMMD loss needs.

In [4]:
# ============================================================
# PART 3: MSUN MODEL
# ============================================================

def build_resnet50():
    try:
        return tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
    except Exception:
        return tvm.resnet50(pretrained=True)


def cbr(cin, cout, k):
    # conv + BN + ReLU. BN and ReLU are an implementation choice; the paper lists convs only.
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, padding=k // 2, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class MSUN(nn.Module):
    def __init__(self, num_classes, g_up_to="layer3", width=128):
        super().__init__()
        r = build_resnet50()
        stages = [r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3]
        in_ch = 1024
        if g_up_to == "layer4":
            stages.append(r.layer4)
            in_ch = 2048
        self.g = nn.Sequential(*stages)

        pool = lambda: nn.AvgPool2d(3, stride=1, padding=1)
        self.h = nn.ModuleList([
            nn.Sequential(cbr(in_ch, width, 1), cbr(width, width, 5), pool()),                  # Sub1
            nn.Sequential(cbr(in_ch, width, 1), cbr(width, width, 3), cbr(width, width, 3), pool()),  # Sub2
            nn.Sequential(cbr(in_ch, width, 1), pool()),                                         # Sub3
            nn.Sequential(pool(), cbr(in_ch, width, 1), pool()),                                 # Sub4
        ])
        self.c = nn.Linear(4 * width, num_classes)

    def forward(self, x, return_feats=False):
        f = self.g(x)
        reps = [F.adaptive_avg_pool2d(branch(f), 1).flatten(1) for branch in self.h]
        logits = self.c(torch.cat(reps, dim=1))
        if return_feats:
            return logits, reps
        return logits


def count_params(model):
    return sum(p.numel() for p in model.parameters())


_probe = MSUN(NUM_CLASSES, G_UP_TO, BRANCH_WIDTH)
_n = count_params(_probe)
print(f"MSUN ({G_UP_TO}, width {BRANCH_WIDTH}): {_n:,} params | FP32 ~= {_n*4/1024**2:.2f} MB")
with torch.no_grad():
    _lg, _rp = _probe.eval()(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE), return_feats=True)
print(f"logits {tuple(_lg.shape)} | representation vectors {[tuple(r.shape) for r in _rp]}")
del _probe, _lg, _rp

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


MSUN (layer3, width 128): 9,775,427 params | FP32 ~= 37.29 MB
logits (2, 3) | representation vectors [(2, 128), (2, 128), (2, 128), (2, 128)]


## Part 4 - Losses and utilities

`lmmd_loss` follows the DSAN-style estimator that MSUN builds on. Each class gets a normalized
weight vector. Source weights are one-hot. Target weights are the softmax probabilities. Only classes
present in both batches contribute. The entropy term is Eq. 10 of the paper.

In [5]:
# ============================================================
# PART 4: LOSSES AND UTILITIES
# ============================================================

def gaussian_kernel_matrix(x, y, kernel_mul=2.0, kernel_num=5):
    total = torch.cat([x, y], dim=0)
    n = total.size(0)
    l2 = (total.unsqueeze(0) - total.unsqueeze(1)).pow(2).sum(2)
    bandwidth = l2.detach().sum() / (n * n - n)
    bandwidth = bandwidth / (kernel_mul ** (kernel_num // 2))
    return sum(torch.exp(-l2 / (bandwidth * (kernel_mul ** i) + 1e-8)) for i in range(kernel_num))


def lmmd_loss(fs, ft, ys, t_prob, num_classes):
    # fs, ft: source / target representation vectors (B, D). ys: source labels (B,).
    # t_prob: detached target softmax (B, C). Returns the class-weighted MMD (Eq. 9).
    bs = fs.size(0)
    K = gaussian_kernel_matrix(fs, ft)
    Kss, Ktt, Kst = K[:bs, :bs], K[bs:, bs:], K[:bs, bs:]

    s_onehot = F.one_hot(ys, num_classes).float()
    s_sum = s_onehot.sum(0)
    t_sum = t_prob.sum(0)
    mask = (s_sum > 0) & (t_sum > 1e-6)
    if int(mask.sum()) == 0:
        return fs.new_zeros(())

    S = s_onehot[:, mask] / s_sum[mask]
    T = t_prob[:, mask] / t_sum[mask]
    n_cls = mask.sum().float()
    w_ss = S @ S.t() / n_cls
    w_tt = T @ T.t() / n_cls
    w_st = S @ T.t() / n_cls
    return (w_ss * Kss + w_tt * Ktt - 2.0 * w_st * Kst).sum()


def entropy_loss(prob):
    return -(prob * torch.log(prob + 1e-8)).sum(1).mean()


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop = 0, None, False

    def __call__(self, score):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True
        return False


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    tot, preds, labs = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds),
            f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs))


def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


cls_criterion = nn.CrossEntropyLoss()
print("Utilities ready: lmmd_loss, entropy_loss, EarlyStopping, evaluate, bootstrap_ci")

Utilities ready: lmmd_loss, entropy_loss, EarlyStopping, evaluate, bootstrap_ci


## Part 5 - One training run

Each iteration takes one source batch and one target batch. They go through the network in
separate forward passes, as in the DSAN/MRAN code that MSUN builds on. The learning rate follows
`lr0 / (1 + 10p)^0.75` and lambda follows the paper's progressive schedule, both driven by the
training progress `p` in [0, 1].

Checkpoint selection mirrors the other baselines: a 5-epoch moving average of target-val accuracy,
patience 15. Set `SELECTION = "last"` for strict UDA, which never looks at target labels.

In [6]:
# ============================================================
# PART 5: TRAIN ONE RUN
# ============================================================

def make_param_groups(model):
    g_params = list(model.g.parameters())
    new_params = list(model.h.parameters()) + list(model.c.parameters())
    return [
        {'params': g_params,   'base_lr': LR_G,               'lr': LR_G},
        {'params': new_params, 'base_lr': LR_G * LR_NEW_MULT, 'lr': LR_G * LR_NEW_MULT},
    ]


def train_one_run(seed, epochs=EPOCHS, selection=SELECTION, verbose=True, tag="run"):
    set_seed(seed)
    model = MSUN(NUM_CLASSES, G_UP_TO, BRANCH_WIDTH).to(DEVICE)
    opt = optim.SGD(make_param_groups(model), momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    stopper = EarlyStopping(patience=PATIENCE)

    total_iters = epochs * len(src_train_loader)
    tgt_iter = cycle(tgt_train_loader)
    step = 0
    raw_hist, best_sm = [], 0.0
    best_state = copy.deepcopy(model.state_dict())
    t0 = time.time()

    if verbose:
        print("=" * 90)
        print(f"  [{tag}] seed={seed} | epochs<={epochs} | selection={selection}")
        print("=" * 90)

    for ep in range(1, epochs + 1):
        model.train()
        n_seen, n_correct = 0, 0
        s_ce = s_lm = s_en = 0.0
        for xs, ys in src_train_loader:
            p = step / max(total_iters, 1)
            lam = LAMBDA_MAX * (2.0 / (1.0 + math.exp(-LAMBDA_ALPHA * p)) - 1.0)
            for pg in opt.param_groups:
                pg['lr'] = pg['base_lr'] / (1.0 + 10.0 * p) ** 0.75

            xt, _ = next(tgt_iter)                    # target labels are NOT used
            xs, ys, xt = xs.to(DEVICE), ys.to(DEVICE), xt.to(DEVICE)

            ls, fs = model(xs, return_feats=True)
            lt, ft = model(xt, return_feats=True)

            ce = F.cross_entropy(ls, ys)
            t_prob = F.softmax(lt, dim=1)
            lm = sum(lmmd_loss(a, b, ys, t_prob.detach(), NUM_CLASSES) for a, b in zip(fs, ft))
            en = entropy_loss(t_prob)
            loss = ce + lam * lm + GAMMA * en

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            step += 1

            n_correct += (ls.argmax(1) == ys).sum().item()
            n_seen += ys.size(0)
            s_ce += ce.item() * ys.size(0)
            s_lm += float(lm) * ys.size(0)
            s_en += en.item() * ys.size(0)

        _, sv_acc, _, _, _     = evaluate(model, src_val_loader, cls_criterion, DEVICE)
        _, tv_acc, tv_f1, _, _ = evaluate(model, tgt_val_loader, cls_criterion, DEVICE)
        raw_hist.append(tv_acc)
        sm = float(np.mean(raw_hist[-SMOOTH_WINDOW:]))

        if selection == "smoothed_target_val":
            stopper(sm)
            if sm > best_sm:
                best_sm = sm
                best_state = copy.deepcopy(model.state_dict())

        if verbose and (ep % 5 == 0 or ep == 1):
            print(f"Ep {ep:3d}/{epochs} | ce={s_ce/n_seen:.3f} lmmd={s_lm/n_seen:.4f} "
                  f"ent={s_en/n_seen:.3f} lam={lam:.2f} | train_acc={n_correct/n_seen:.3f} | "
                  f"SRC-VAL={sv_acc:.3f} | TGT-VAL={tv_acc:.3f} (sm={sm:.3f}) f1={tv_f1:.3f} | "
                  f"{time.time()-t0:.0f}s")

        if selection == "smoothed_target_val" and stopper.early_stop:
            if verbose:
                print(f"\nEarly stop at epoch {ep}")
            break

    if selection == "smoothed_target_val":
        model.load_state_dict(best_state)
    if verbose:
        print(f"Done in {time.time()-t0:.0f}s | best smoothed TGT-VAL = {best_sm:.4f}")
    return model, raw_hist, best_sm

## Part 6 - Run, evaluate on the held-out target test set

The target test set is touched once per run. The notebook also saves per-image predictions
for every run. They allow a paired test (for example McNemar) against your other models, because
all models are evaluated on the same 79 images.

In [7]:
# ============================================================
# PART 6: RUN + EVALUATE
# ============================================================
rows = []
for s in SEEDS:
    print(f"\n{'#'*90}\n# SEED {s}  (MSUN)\n{'#'*90}")
    m, hist, best_sm = train_one_run(s, tag=f"msun_seed{s}")

    _, src_test_acc, src_test_f1, _, _   = evaluate(m, src_test_loader, cls_criterion, DEVICE)
    _, tgt_test_acc, tgt_test_f1, tp, tl = evaluate(m, tgt_test_loader, cls_criterion, DEVICE)
    bmean, blo, bhi = bootstrap_ci(tl, tp)

    rows.append(dict(seed=s, src_test_acc=src_test_acc, src_test_f1=src_test_f1,
                     tgt_test_acc=tgt_test_acc, tgt_test_f1=tgt_test_f1,
                     boot_mean=bmean, ci_lo=blo, ci_hi=bhi))
    print(f"  seed {s}: SRC-TEST={src_test_acc:.4f}  TGT-TEST={tgt_test_acc:.4f} "
          f"(95% CI [{blo:.4f}, {bhi:.4f}])  TGT-F1={tgt_test_f1:.4f}")

    # per-image predictions on the target test set (loader is not shuffled, order is stable)
    pd.DataFrame({'path': tgt_test_idx.img_paths, 'label': tl, 'pred': tp}).to_csv(
        f"{TABLES_DIR}/msun_tgt_test_predictions_seed{s}.csv", index=False)

    torch.save(m.state_dict(), f"{MODELS_DIR}/msun_seed{s}.pth")
    del m
    torch.cuda.empty_cache()

msun_df = pd.DataFrame(rows)
msun_df.to_csv(f"{TABLES_DIR}/msun_runs.csv", index=False)
print("\n" + msun_df.to_string(index=False))


##########################################################################################
# SEED 42  (MSUN)
##########################################################################################
  [msun_seed42] seed=42 | epochs<=60 | selection=smoothed_target_val
Ep   1/60 | ce=0.133 lmmd=4.5181 ent=0.375 lam=0.08 | train_acc=0.944 | SRC-VAL=0.950 | TGT-VAL=0.696 (sm=0.696) f1=0.686 | 78s
Ep   5/60 | ce=0.149 lmmd=3.1057 ent=0.444 lam=0.39 | train_acc=0.985 | SRC-VAL=0.993 | TGT-VAL=0.785 (sm=0.754) f1=0.778 | 325s
Ep  10/60 | ce=0.232 lmmd=1.0371 ent=0.574 lam=0.68 | train_acc=0.981 | SRC-VAL=0.885 | TGT-VAL=0.797 (sm=0.790) f1=0.790 | 630s
Ep  15/60 | ce=1.059 lmmd=0.0368 ent=1.045 lam=0.85 | train_acc=0.472 | SRC-VAL=0.472 | TGT-VAL=0.304 (sm=0.304) f1=0.142 | 936s
Ep  20/60 | ce=1.060 lmmd=0.0082 ent=1.046 lam=0.93 | train_acc=0.472 | SRC-VAL=0.472 | TGT-VAL=0.304 (sm=0.304) f1=0.142 | 1240s
Ep  25/60 | ce=1.060 lmmd=0.0044 ent=1.049 lam=0.97 | train_acc=0.471 | SRC-VAL=0.472

## Part 7 - Size and training-time memory

Size here is a parameter-count estimate, the same convention used for AgroNet: INT8 estimate =
parameters x 1 byte. No quantized file is produced. The memory probe runs a short training and reads
PyTorch's peak GPU memory counter, the same method used for AgroNet and your system.

In [8]:
# ============================================================
# PART 7: SIZE + TRAINING MEMORY
# ============================================================
_m = MSUN(NUM_CLASSES, G_UP_TO, BRANCH_WIDTH)
n_params = count_params(_m)
size_fp32_mb = n_params * 4 / 1024 ** 2
size_int8_est_mb = n_params / 1024 ** 2
del _m
print(f"Params            : {n_params:,}")
print(f"FP32 size         : {size_fp32_mb:.2f} MB")
print(f"INT8 size (est.)  : {size_int8_est_mb:.2f} MB  (params x 1 byte, not a measured file)")

peak_mem_mb, probe_seconds = None, None
if DEVICE == "cuda":
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _t0 = time.time()
    _pm, _, _ = train_one_run(SEEDS[0], epochs=PROBE_EPOCHS, verbose=False, tag="memory_probe")
    torch.cuda.synchronize()
    probe_seconds = time.time() - _t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    del _pm
    torch.cuda.empty_cache()
    print(f"Peak GPU memory   : {peak_mem_mb:.1f} MB  ({PROBE_EPOCHS}-epoch probe, {probe_seconds:.0f}s)")
else:
    print("No GPU: memory probe skipped.")

Params            : 9,775,427
FP32 size         : 37.29 MB
INT8 size (est.)  : 9.32 MB  (params x 1 byte, not a measured file)
Peak GPU memory   : 2807.6 MB  (2-epoch probe, 124s)


## Part 8 - Result for the comparison table

In [9]:
# ============================================================
# PART 8: RESULT + LATEX ROW
# ============================================================
if len(SEEDS) == 1:
    r = rows[0]
    acc_txt = (f"{r['tgt_test_acc']*100:.2f}~[{r['ci_lo']*100:.2f}, {r['ci_hi']*100:.2f}]"
               r"$^{\dagger}$")
else:
    acc_txt = (f"{msun_df.tgt_test_acc.mean()*100:.2f} $\\pm$ {msun_df.tgt_test_acc.std()*100:.2f}")

result = dict(
    method="MSUN (Wu et al., 2023), this reproduction",
    seeds=SEEDS,
    params=n_params,
    size_fp32_mb=round(size_fp32_mb, 2),
    size_int8_est_mb=round(size_int8_est_mb, 2),
    src_test_acc=round(float(msun_df.src_test_acc.mean()), 4),
    tgt_test_acc=round(float(msun_df.tgt_test_acc.mean()), 4),
    tgt_ci=(round(float(msun_df.ci_lo.iloc[0]), 4), round(float(msun_df.ci_hi.iloc[0]), 4)) if len(SEEDS) == 1 else None,
    train_peak_mem_mb=None if peak_mem_mb is None else round(peak_mem_mb, 1),
)
for k, v in result.items():
    print(f"{k:18s}: {v}")

latex_row = (r"MSUN~\cite{wu2023msun} & ResNet-50 + Inception adaptation & "
             r"LMMD + entropy min. & " + acc_txt + f" & {size_int8_est_mb:.2f} MB \\\\")
print("\nLaTeX row (INT8-estimate size, same convention as the AgroNet row):\n")
print(latex_row)

pd.DataFrame([result]).to_csv(f"{TABLES_DIR}/msun_result.csv", index=False)

method            : MSUN (Wu et al., 2023), this reproduction
seeds             : [42]
params            : 9775427
size_fp32_mb      : 37.29
size_int8_est_mb  : 9.32
src_test_acc      : 0.8989
tgt_test_acc      : 0.7722
tgt_ci            : (0.6835, 0.8608)
train_peak_mem_mb : 2807.6

LaTeX row (INT8-estimate size, same convention as the AgroNet row):

MSUN~\cite{wu2023msun} & ResNet-50 + Inception adaptation & LMMD + entropy min. & 77.22~[68.35, 86.08]$^{\dagger}$ & 9.32 MB \\


## Caveats to state in the manuscript

1. **Reconstruction, not original code.** The paper gives no code link. Unspecified details were
   assumed: the split point of `g`, the branch width, the base learning rate, the weight decay and the
   batch size. They are listed in the config cell. No tuning was done on this split.
2. **No tuning of gamma.** The authors grid-searched gamma on their own data. Here gamma is fixed at
   0.1, the value they report as best.
3. **Epochs.** The paper trains 200 epochs. This run uses at most 60 with early stopping, to match the
   protocol of the other baselines.
4. **Model selection uses target-val labels.** This matches your other models. MSUN is nominally
   unsupervised. Set `SELECTION = "last"` for a strict UDA number and say which one you report.
5. **Target labels in training: none.** Only target-train images are used, without labels.
   Target-val labels are used for checkpoint selection only.
6. **Not comparable to the paper's 56.06%.** That number is a 27-class PlantVillage to PlantDoc
   transfer. This notebook uses your 3-class split.
7. **Size is an estimate.** It is parameters x 1 byte. It is not a measured INT8 file.
8. **Single seed (42).** Report the bootstrap CI. Do not put it next to a multi-seed mean +/- std
   without the footnote.
9. **Input size.** 224 px, as in the paper and as in the EffNet baseline. Your own system uses a
   different resolution.

## Part 9 - Real INT8 quantization (same route as MicroBrainNet-DA / AgroNet comparison; no ONNX)

Everything above is unchanged. This part **adds** a real, measured INT8 export of the trained MSUN model(s) and replaces the
`params x 1 byte` estimate of Part 7 with a measured `.tflite` file and a measured INT8 accuracy.

**Route (identical to the other notebooks):**
1. Reload the saved checkpoint(s) `models/msun_seed{s}.pth` (Part 6 deletes the in-memory model) and re-evaluate them in FP32. This also verifies that the checkpoint reproduces the Part 6 numbers.
2. Rebuild MSUN natively in TF/Keras and copy the PyTorch weights layer by layer (shape-checked by `set_weights`).
3. **Parity gate**: Keras vs PyTorch logits on real images; a weight-copy bug aborts the export instead of silently quantizing a wrong model.
4. `tf.lite.TFLiteConverter`, `Optimize.DEFAULT`, representative dataset, `TFLITE_BUILTINS_INT8`, int8 input and output.
5. ImageNet mean/std is baked into the graph, so the TFLite model takes raw [0,1] pixels.
6. Calibration = up to 400 **source-val** + all **target-val** images (validation splits only; `tgt_test` is never used).
7. The real TFLite interpreter is run on the 79 `tgt_test` images (and the 564 `src_test` images) with the input quantized by the model's own scale / zero-point.

**Translation details that matter for correctness (all checked by the parity gate):**
- `AvgPool2d(3, stride 1, padding 1)` in PyTorch divides by 9 at the borders (`count_include_pad=True`); Keras `same` average pooling divides by the number of valid cells. It is therefore rebuilt as `ZeroPadding2D(1)` + `valid` pooling.
- `MaxPool2d(3, stride 2, padding 1)`: TF `same` pads asymmetrically (0 before, 1 after on 112 px). It is rebuilt as `ZeroPadding2D(1)` + `valid`; zero padding equals PyTorch's -inf padding here because the stem output is post-ReLU (>= 0).
- Every stride-2 conv (7x7 stem, ResNet bottleneck 3x3, 1x1 downsample) uses explicit symmetric padding, as in PyTorch.
- The ResNet-50 `Bottleneck` blocks (with 1x1 downsample shortcuts) are translated explicitly; the four substructures use `conv + BN + ReLU` exactly as defined in Part 3.
- In inference MSUN is `g -> four substructures -> global pooling -> concat -> FC` (the LMMD / entropy terms are training-only).

TensorFlow is imported only after all PyTorch evaluation and all DataLoader use in this notebook, and it is hidden from the GPU.

In [10]:
# ============================================================
# 9A: RELOAD CHECKPOINT(S) + FP32 REFERENCE (PyTorch) -- done BEFORE TensorFlow is imported
# ============================================================
fp32_ref = {}
for s in SEEDS:
    ckpt = f"{MODELS_DIR}/msun_seed{s}.pth"
    m_ = MSUN(NUM_CLASSES, G_UP_TO, BRANCH_WIDTH)
    m_.load_state_dict(torch.load(ckpt, map_location="cpu"))
    m_.to(DEVICE).eval()
    _, sa_, _, _, _      = evaluate(m_, src_test_loader, cls_criterion, DEVICE)
    _, ta_, _, tp_, tl_  = evaluate(m_, tgt_test_loader, cls_criterion, DEVICE)
    ref_row = msun_df[msun_df.seed == s].iloc[0]
    same = (abs(sa_ - ref_row.src_test_acc) < 1e-9) and (abs(ta_ - ref_row.tgt_test_acc) < 1e-9)
    print(f"seed {s}: reloaded {ckpt}")
    print(f"  FP32 SRC-TEST={sa_:.4f}  TGT-TEST={ta_:.4f}  | identical to Part 6 numbers: {same}")
    fp32_ref[s] = dict(model=m_.cpu().eval(), src_acc=sa_, tgt_acc=ta_, tp=tp_, tl=tl_)
    del m_
torch.cuda.empty_cache()

seed 42: reloaded /kaggle/working/msun_baseline/models/msun_seed42.pth
  FP32 SRC-TEST=0.8989  TGT-TEST=0.7722  | identical to Part 6 numbers: True


In [11]:
# ============================================================
# 9B: TENSORFLOW SETUP + RAW UINT8 ARRAYS + HELPERS
# ============================================================
import gc, traceback
from collections import Counter

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
try:
    tf.config.set_visible_devices([], "GPU")     # TF is used for CPU export / INT8 eval only
except Exception as _e:
    print("note: could not hide GPUs from TensorFlow:", _e)
tf.get_logger().setLevel("ERROR")
print("TensorFlow:", tf.__version__)

CAL_N_SRC       = 400      # calibration: up to 400 source-VAL images + ALL target-VAL images
CAL_SEED        = 0
INT8_EVAL_SRC   = True     # also run the INT8 model on the 564 source-test images (False to skip)
TFLITE_THREADS  = 4
PARITY_N        = 32
PARITY_FAIL_REL = 0.05     # abort the export if max|dlogit| > 5% of the largest |logit| (weight-copy bug)


def load_raw_arrays(idx_obj, name=""):
    # Same ops as eval_tf (Resize to IMG_SIZE) minus ToTensor/Normalize. ToTensor is x/255, so uint8 is lossless.
    rs = transforms.Resize((IMG_SIZE, IMG_SIZE))
    arr = np.empty((len(idx_obj.img_paths), IMG_SIZE, IMG_SIZE, 3), np.uint8)
    for i, pth in enumerate(idx_obj.img_paths):
        arr[i] = np.asarray(rs(Image.open(pth).convert("RGB")), dtype=np.uint8)
    print(f"  raw[{name:8s}] {arr.shape}  {arr.nbytes/1024**2:6.1f} MB")
    return arr


RAW = dict(
    src_val=load_raw_arrays(src_val_idx, "src_val"),
    src_test=load_raw_arrays(src_test_idx, "src_test"),
    tgt_val=load_raw_arrays(tgt_val_idx, "tgt_val"),
    tgt_test=load_raw_arrays(tgt_test_idx, "tgt_test"),
    src_test_labels=np.array(src_test_idx.labels),
    tgt_test_labels=np.array(tgt_test_idx.labels),
)


def make_calibration(n_src=CAL_N_SRC, seed=CAL_SEED):
    # Representative set: source-VAL (subsampled) + ALL target-VAL. No test image is used.
    rng = np.random.RandomState(seed)
    pick = rng.permutation(len(RAW["src_val"]))[:n_src]
    cal = np.concatenate([RAW["src_val"][pick], RAW["tgt_val"]], axis=0)
    return cal[rng.permutation(len(cal))]


@torch.no_grad()
def torch_logits_from_raw(model, arr_u8, bs=16):
    model.eval()
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    outs = []
    for i in range(0, len(arr_u8), bs):
        x = torch.from_numpy(arr_u8[i:i + bs]).permute(0, 3, 1, 2).float() / 255.0
        outs.append(model((x - mean) / std).float())
    return torch.cat(outs).numpy()


_c = make_calibration()
print(f"calibration set: {_c.shape[0]} images ({min(CAL_N_SRC, len(RAW['src_val']))} source-val + {len(RAW['tgt_val'])} target-val)")
del _c

TensorFlow: 2.20.0
  raw[src_val ] (564, 224, 224, 3)    81.0 MB
  raw[src_test] (564, 224, 224, 3)    81.0 MB
  raw[tgt_val ] (79, 224, 224, 3)    11.3 MB
  raw[tgt_test] (79, 224, 224, 3)    11.3 MB
calibration set: 479 images (400 source-val + 79 target-val)


In [12]:
# ============================================================
# 9C: TORCH -> KERAS TRANSLATOR (weights copied layer by layer)
# ============================================================
L = tf.keras.layers
_uid = {"n": 0}

def _nm(prefix):
    _uid["n"] += 1
    return f"{prefix}_{_uid['n']}"

def _np(t):
    return t.detach().cpu().numpy().astype(np.float32)

def _to_int(v):
    return int(v[0]) if isinstance(v, (tuple, list)) else int(v)


def _conv(x, m, name):
    kh, kw = m.kernel_size
    assert kh == kw and tuple(m.dilation) == (1, 1), "only square, non-dilated convs are supported"
    k, s, pad, g = kh, m.stride[0], m.padding[0], m.groups
    assert pad in (0, k // 2), f"unsupported padding {pad} for k={k}"
    assert g == 1, "grouped convs are not used by MSUN"
    if s > 1 and pad > 0:                      # explicit symmetric pad, as PyTorch (TF 'same' is asymmetric)
        x = L.ZeroPadding2D(pad, name=_nm(name + "_pad"))(x)
        padding = "valid"
    else:
        padding = "same" if pad > 0 else "valid"
    use_bias = m.bias is not None
    layer = L.Conv2D(m.out_channels, k, strides=s, padding=padding, use_bias=use_bias, name=_nm(name))
    y = layer(x)
    ws = [_np(m.weight).transpose(2, 3, 1, 0)]  # (Cout,Cin,kh,kw) -> (kh,kw,Cin,Cout)
    if use_bias:
        ws.append(_np(m.bias))
    layer.set_weights(ws)
    return y


def _bn(x, m, name):
    layer = L.BatchNormalization(epsilon=m.eps, name=_nm(name))
    y = layer(x)
    layer.set_weights([_np(m.weight), _np(m.bias), _np(m.running_mean), _np(m.running_var)])
    return y


def _dense(x, W_out_in, b, name):
    layer = L.Dense(W_out_in.shape[0], name=_nm(name))
    y = layer(x)
    layer.set_weights([np.ascontiguousarray(W_out_in.T), np.ascontiguousarray(b)])
    return y


def _bottleneck(x, m, name):
    # torchvision ResNet Bottleneck: 1x1 -> 3x3(stride) -> 1x1, BN after each, ReLU, shortcut (optional 1x1 downsample).
    idt = x
    y = L.ReLU(name=_nm(name + "_r1"))(_bn(_conv(x, m.conv1, name + "_c1"), m.bn1, name + "_b1"))
    y = L.ReLU(name=_nm(name + "_r2"))(_bn(_conv(y, m.conv2, name + "_c2"), m.bn2, name + "_b2"))
    y = _bn(_conv(y, m.conv3, name + "_c3"), m.bn3, name + "_b3")
    if m.downsample is not None:
        idt = _tr(x, m.downsample, name + "_ds")
    return L.ReLU(name=_nm(name + "_r3"))(L.Add(name=_nm(name + "_add"))([y, idt]))


def _tr(x, m, name):
    cn = m.__class__.__name__
    if isinstance(m, nn.Conv2d):
        return _conv(x, m, name)
    if isinstance(m, nn.BatchNorm2d):
        return _bn(x, m, name)
    if isinstance(m, nn.ReLU):
        return L.ReLU(name=_nm(name))(x)
    if isinstance(m, nn.MaxPool2d):
        k, s, p = _to_int(m.kernel_size), _to_int(m.stride), _to_int(m.padding)
        assert not m.ceil_mode
        # input is post-ReLU (>= 0), so zero padding == PyTorch's -inf padding; TF 'same' would pad asymmetrically
        if p > 0:
            x = L.ZeroPadding2D(p, name=_nm(name + "_pad"))(x)
        return L.MaxPooling2D(k, strides=s, padding="valid", name=_nm(name))(x)
    if isinstance(m, nn.AvgPool2d):
        k, s, p = _to_int(m.kernel_size), _to_int(m.stride), _to_int(m.padding)
        assert not m.ceil_mode and m.count_include_pad, "translator assumes count_include_pad=True (PyTorch default)"
        # PyTorch divides by k*k at the borders; Keras 'same' divides by the number of valid cells -> pad explicitly
        if p > 0:
            x = L.ZeroPadding2D(p, name=_nm(name + "_pad"))(x)
        return L.AveragePooling2D(k, strides=s, padding="valid", name=_nm(name))(x)
    if isinstance(m, (nn.Dropout, nn.Identity)):
        return x
    if cn == "Bottleneck":
        return _bottleneck(x, m, name)
    if isinstance(m, nn.Sequential):
        for i, c in enumerate(m):
            x = _tr(x, c, f"{name}_{i}")
        return x
    raise NotImplementedError(f"no Keras translation for {cn}")


def build_keras_msun(tm, img_size):
    # Inference path of MSUN: g -> 4 substructures -> global avg pool each -> concat -> FC.
    _uid["n"] = 0
    tm = tm.cpu().eval()
    inp = L.Input(shape=(img_size, img_size, 3), batch_size=1, name="input")
    sc = (1.0 / np.array(IMAGENET_STD, np.float32)).astype(np.float32)
    of = (-np.array(IMAGENET_MEAN, np.float32) * sc).astype(np.float32)
    x = L.Lambda(lambda t: t * sc + of, name="input_norm")(inp)          # (x - mean) / std baked into the graph
    f = _tr(x, tm.g, "g")
    reps = []
    for bi, branch in enumerate(tm.h):
        y = _tr(f, branch, f"h{bi}")
        reps.append(L.GlobalAveragePooling2D(name=f"gap{bi}")(y))
    z = L.Concatenate(axis=-1, name="rep_cat")(reps)
    out = _dense(z, _np(tm.c.weight), _np(tm.c.bias), "head")
    return tf.keras.Model(inp, out, name="MSUN_deploy")


print("Keras translator ready.")

Keras translator ready.


In [13]:
# ============================================================
# 9D: TFLITE CONVERSION (FP32 + full INT8), INTERPRETER EVALUATION, AUDIT
# ============================================================

def _make_rep_gen(cal_u8):
    def gen():
        for i in range(cal_u8.shape[0]):
            yield [cal_u8[i:i + 1].astype(np.float32) / 255.0]       # model input is raw [0,1]
    return gen


def _convert(km, cal_u8=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(km)
    if cal_u8 is not None:                                           # identical settings to the other notebooks
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = _make_rep_gen(cal_u8)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()


def tflite_logits(model_bytes, arr_u8, int8, threads=TFLITE_THREADS):
    # REAL TFLite interpreter. INT8: input quantized with the model's own scale/zero-point, int8 output dequantized.
    it = tf.lite.Interpreter(model_content=model_bytes, num_threads=threads)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    (s_i, z_i), (s_o, z_o) = ind["quantization"], outd["quantization"]
    if int8:
        assert ind["dtype"] == np.int8 and outd["dtype"] == np.int8, "model I/O is not int8"
    out = np.zeros((len(arr_u8), outd["shape"][-1]), np.float32)
    for i in range(len(arr_u8)):
        a = arr_u8[i:i + 1].astype(np.float32) / 255.0
        if int8:
            a = np.clip(np.round(a / s_i + z_i), -128, 127).astype(np.int8)
        it.set_tensor(ind["index"], a)
        it.invoke()
        o = it.get_tensor(outd["index"])[0]
        out[i] = (o.astype(np.float32) - z_o) * s_o if int8 else o
    return out


def audit_tflite(model_bytes, label):
    it = tf.lite.Interpreter(model_content=model_bytes)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    cnt, big_float = Counter(), []
    for d in it.get_tensor_details():
        dt = np.dtype(d["dtype"])
        n = int(np.prod(d["shape"])) if len(d["shape"]) else 1
        cnt[dt.name] += 1
        if dt == np.float32 and n >= 1000:
            big_float.append((d["name"][:60], n))
    print(f"\n--- INT8 audit: {label} ---")
    print(f"file size        : {len(model_bytes)/1024:.1f} KB")
    print(f"input            : dtype={np.dtype(ind['dtype']).name} shape={[int(v) for v in ind['shape']]} (scale, zero_point)={ind['quantization']}")
    print(f"output           : dtype={np.dtype(outd['dtype']).name} shape={[int(v) for v in outd['shape']]} (scale, zero_point)={outd['quantization']}")
    print(f"tensors by dtype : {dict(cnt)}")
    print("float32 tensors >=1000 elements:", big_float[:6] if big_float else "none (no un-quantized weights / activations)")
    try:
        ops = Counter(d["op_name"] for d in it._get_ops_details())
        print(f"ops ({sum(ops.values())} nodes): {dict(sorted(ops.items()))}")
        bad = [o for o in ops if "flex" in o.lower() or "custom" in o.lower()]
        print("Flex/custom ops  :", bad if bad else "none")
    except Exception as e:
        print("op list not available in this TF build:", e)


def export_msun_int8(tm_cpu, tag):
    tf.keras.backend.clear_session(); gc.collect()
    t0 = time.time()
    km = build_keras_msun(tm_cpu, IMG_SIZE)                 # translation bugs are NOT caught: they must be loud
    arr = RAW["tgt_val"][:PARITY_N]
    ref = torch_logits_from_raw(tm_cpu, arr)
    got = np.concatenate([km(arr[i:i + 1].astype(np.float32) / 255.0, training=False).numpy() for i in range(len(arr))], 0)
    mad = float(np.abs(ref - got).max()); rel = mad / (float(np.abs(ref).max()) + 1e-6)
    agree = float((ref.argmax(1) == got.argmax(1)).mean())
    print(f"  [{tag}] Keras params {km.count_params():,} | parity vs PyTorch: max|dlogit|={mad:.2e} (rel {rel:.2e}), "
          f"top-1 agreement {agree*100:.1f}%")
    if rel > PARITY_FAIL_REL:
        raise RuntimeError(f"Keras model does not reproduce the PyTorch model (rel diff {rel:.3f}) - "
                           f"weight copy / layout bug, refusing to quantize.")
    cal = make_calibration()
    fp32_bytes = _convert(km, None)
    int8_bytes = _convert(km, cal)
    print(f"  [{tag}] converted in {time.time()-t0:.0f}s | calibration images: {len(cal)} | "
          f"FP32 tflite {len(fp32_bytes)/1024:.1f} KB | INT8 tflite {len(int8_bytes)/1024:.1f} KB")
    return dict(fp32_bytes=fp32_bytes, int8_bytes=int8_bytes, parity_rel=rel, parity_agree=agree)


print("TFLite export / evaluation helpers ready.")

TFLite export / evaluation helpers ready.


In [14]:
# ============================================================
# 9E: EXPORT + EVALUATE (one iteration per entry in SEEDS)
# ============================================================
q_rows = []
for s in SEEDS:
    print(f"\n{'#'*90}\n# MSUN seed {s}: INT8 export + evaluation\n{'#'*90}")
    ref = fp32_ref[s]
    row = dict(seed=s, int8_ok=False)
    try:
        res = export_msun_int8(copy.deepcopy(ref["model"]).cpu().eval(), f"msun_seed{s}")
        with open(f"{MODELS_DIR}/msun_seed{s}_fp32.tflite", "wb") as f: f.write(res["fp32_bytes"])
        with open(f"{MODELS_DIR}/msun_seed{s}_int8.tflite", "wb") as f: f.write(res["int8_bytes"])

        yt = RAW["tgt_test_labels"]
        assert np.array_equal(yt, ref["tl"]), "raw target-test order differs from the loader order"
        lg_f = tflite_logits(res["fp32_bytes"], RAW["tgt_test"], False)
        lg_i = tflite_logits(res["int8_bytes"], RAW["tgt_test"], True)
        pf, pi = lg_f.argmax(1), lg_i.argmax(1)
        ci_i = bootstrap_ci(yt, pi)
        acc_fp32, acc_tfl, acc_i8 = accuracy_score(yt, ref["tp"]), accuracy_score(yt, pf), accuracy_score(yt, pi)

        src_i8 = float("nan")
        if INT8_EVAL_SRC:
            lg_s = tflite_logits(res["int8_bytes"], RAW["src_test"], True)
            src_i8 = accuracy_score(RAW["src_test_labels"], lg_s.argmax(1))

        row.update(int8_ok=True,
                   fp32_src_acc=ref["src_acc"], fp32_tgt_acc=acc_fp32, tflite_fp32_tgt_acc=acc_tfl,
                   int8_tgt_acc=acc_i8, int8_ci_lo=ci_i[1], int8_ci_hi=ci_i[2],
                   int8_tgt_f1=f1_score(yt, pi, average="weighted"), int8_src_acc=src_i8,
                   acc_drop_pts=(acc_fp32 - acc_i8) * 100, agree_with_fp32=float((pi == ref["tp"]).mean()),
                   tflite_fp32_kb=len(res["fp32_bytes"]) / 1024, int8_kb=len(res["int8_bytes"]) / 1024,
                   parity_rel=res["parity_rel"])
        pd.DataFrame({"path": tgt_test_idx.img_paths, "label": yt, "pred_fp32_torch": ref["tp"],
                      "pred_tflite_fp32": pf, "pred_int8": pi}).to_csv(
            f"{TABLES_DIR}/msun_tgt_test_predictions_int8_seed{s}.csv", index=False)
        print(f"  seed {s}: target-test  FP32(PyTorch)={acc_fp32:.4f}  FP32(TFLite)={acc_tfl:.4f}  INT8(TFLite)={acc_i8:.4f}  "
              f"95% CI [{ci_i[1]:.4f}, {ci_i[2]:.4f}]")
        if INT8_EVAL_SRC:
            print(f"  seed {s}: source-test  FP32(PyTorch)={ref['src_acc']:.4f}  INT8(TFLite)={src_i8:.4f}")
        print(f"  seed {s}: drop FP32->INT8 = {row['acc_drop_pts']:+.2f} pts | INT8-vs-FP32 agreement = "
              f"{row['agree_with_fp32']*100:.1f}% | INT8 file = {row['int8_kb']:.1f} KB")
        audit_tflite(res["int8_bytes"], f"MSUN seed {s} (msun_seed{s}_int8.tflite)")
        del res
    except Exception as e:                       # the trained checkpoint is safe on disk; never lose it to an export error
        print(f"  !! INT8 export FAILED for seed {s}: {type(e).__name__}: {e}")
        traceback.print_exc()
    q_rows.append(row)
    gc.collect()

quant_df = pd.DataFrame(q_rows)
quant_df.to_csv(f"{TABLES_DIR}/msun_quantization_results.csv", index=False)
print("\n" + "=" * 100)
cols = [c for c in ["seed", "int8_ok", "fp32_tgt_acc", "tflite_fp32_tgt_acc", "int8_tgt_acc", "acc_drop_pts",
                    "agree_with_fp32", "int8_src_acc", "tflite_fp32_kb", "int8_kb"] if c in quant_df.columns]
print(quant_df[cols].round(4).to_string(index=False))


##########################################################################################
# MSUN seed 42: INT8 export + evaluation
##########################################################################################
  [msun_seed42] Keras params 9,807,811 | parity vs PyTorch: max|dlogit|=2.15e-06 (rel 5.65e-07), top-1 agreement 100.0%
Saved artifact at '/tmp/tmpsmo5jo26'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  132434108392720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108391568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108395600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396176: TensorSpec(shape

W0000 00:00:1793597197.050452      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793597197.050537      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793597197.180586      23 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Saved artifact at '/tmp/tmp3zctv9ko'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  132434108392720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108391568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108395600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108396944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108397136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108395024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108397520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132434108394064: TensorSpec

W0000 00:00:1793597206.983892      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793597206.983919      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


  [msun_seed42] converted in 182s | calibration images: 479 | FP32 tflite 38149.7 KB | INT8 tflite 9990.3 KB


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


  seed 42: target-test  FP32(PyTorch)=0.7722  FP32(TFLite)=0.7722  INT8(TFLite)=0.7342  95% CI [0.6329, 0.8228]
  seed 42: source-test  FP32(PyTorch)=0.8989  INT8(TFLite)=0.8191
  seed 42: drop FP32->INT8 = +3.80 pts | INT8-vs-FP32 agreement = 87.3% | INT8 file = 9990.3 KB

--- INT8 audit: MSUN seed 42 (msun_seed42_int8.tflite) ---
file size        : 9990.3 KB
input            : dtype=int8 shape=[1, 224, 224, 3] (scale, zero_point)=(0.003921568859368563, -128)
output           : dtype=int8 shape=[1, 3] (scale, zero_point)=(0.023218797519803047, -36)
tensors by dtype : {'int8': 140, 'int32': 54}
float32 tensors >=1000 elements: none (no un-quantized weights / activations)
ops (89 nodes): {'ADD': 14, 'AVERAGE_POOL_2D': 5, 'CONCATENATION': 1, 'CONV_2D': 50, 'DELEGATE': 3, 'FULLY_CONNECTED': 1, 'MAX_POOL_2D': 1, 'MEAN': 4, 'MUL': 1, 'PAD': 9}
Flex/custom ops  : none

 seed  int8_ok  fp32_tgt_acc  tflite_fp32_tgt_acc  int8_tgt_acc  acc_drop_pts  agree_with_fp32  int8_src_acc  tflite_fp32_kb

### 9F. Measured INT8 result, LaTeX row, comparison

The LaTeX row below replaces the one from Part 8: the accuracy is the accuracy of the **real INT8 interpreter** on the target test set, and the size is the **measured `.tflite` file**.
The MicroBrainNet-DA row uses the numbers from its own notebook (INT8 TFLite 245.6 KB, 167,131 parameters, 75.95 % target accuracy of the exported INT8 model, CI [0.6582, 0.8481]) — **edit the constants if you re-run it**.

In [15]:
# ============================================================
# 9F: MEASURED SIZE + RESULT + LATEX ROW + COMPARISON
# ============================================================
ok = quant_df[quant_df["int8_ok"].astype(bool)] if "int8_ok" in quant_df else quant_df.iloc[0:0]

if len(ok) == 0:
    print("No INT8 model was exported: no measured size / INT8 accuracy exists (the Part 7 estimate is NOT substituted).")
else:
    int8_kb = float(ok.int8_kb.mean())
    print(f"Params                : {n_params:,}")
    print(f"FP32 size             : {size_fp32_mb:.2f} MB (params x 4 bytes)")
    print(f"INT8 size, old est.   : {size_int8_est_mb:.2f} MB (params x 1 byte, Part 7)")
    print(f"INT8 .tflite MEASURED : {int8_kb:.1f} KB = {int8_kb/1024:.2f} MB   (FP32 .tflite {ok.tflite_fp32_kb.mean():.1f} KB, "
          f"compression {ok.tflite_fp32_kb.mean()/int8_kb:.2f}x)")
    print(f"measured / estimate   : {(int8_kb/1024)/size_int8_est_mb:.2f}x")

    if len(ok) == 1:
        r_ = ok.iloc[0]
        acc_txt_int8 = (f"{r_.int8_tgt_acc*100:.2f}~[{r_.int8_ci_lo*100:.2f}, {r_.int8_ci_hi*100:.2f}]" r"$^{\dagger}$")
    else:
        acc_txt_int8 = f"{ok.int8_tgt_acc.mean()*100:.2f} $\\pm$ {ok.int8_tgt_acc.std()*100:.2f}"
    latex_row_int8 = (r"MSUN~\cite{wu2023msun} & ResNet-50 + Inception adaptation & LMMD + entropy min. & "
                      + acc_txt_int8 + f" & {int8_kb/1024:.2f} MB \\\\")
    print("\nLaTeX row (INT8 accuracy and INT8 size are both MEASURED on the .tflite file):\n")
    print(latex_row_int8)

    result_int8 = dict(
        method="MSUN (Wu et al., 2023), this reproduction, real INT8 TFLite",
        seeds=SEEDS, params=n_params, size_fp32_mb=round(size_fp32_mb, 2),
        int8_tflite_kb_measured=round(int8_kb, 1),
        tgt_test_acc_fp32=round(float(ok.fp32_tgt_acc.mean()), 4),
        tgt_test_acc_int8=round(float(ok.int8_tgt_acc.mean()), 4),
        tgt_ci_int8=(round(float(ok.int8_ci_lo.iloc[0]), 4), round(float(ok.int8_ci_hi.iloc[0]), 4)) if len(ok) == 1 else None,
        src_test_acc_fp32=round(float(ok.fp32_src_acc.mean()), 4),
        src_test_acc_int8=None if ok.int8_src_acc.isna().all() else round(float(ok.int8_src_acc.mean()), 4),
        train_peak_mem_mb=None if peak_mem_mb is None else round(peak_mem_mb, 1),
    )
    for k, v in result_int8.items():
        print(f"{k:26s}: {v}")
    pd.DataFrame([result_int8]).to_csv(f"{TABLES_DIR}/msun_result_int8.csv", index=False)

    # ---- MicroBrainNet-DA numbers: UPDATE if you re-run it ----
    IM = dict(name="MicroBrainNet-DA (v7, exported model)", fp32_acc=0.7595, int8_acc=0.7595, ci_lo=0.6582, ci_hi=0.8481,
              int8_kb=245.6, params=167_131)
    cmp = pd.DataFrame([
        {"Model": "MSUN (this reproduction)", "FP32 target acc": ok.fp32_tgt_acc.mean(), "INT8 target acc": ok.int8_tgt_acc.mean(),
         "INT8 CI low": ok.int8_ci_lo.iloc[0], "INT8 CI high": ok.int8_ci_hi.iloc[0],
         "INT8 .tflite (KB)": int8_kb, "Total params": n_params},
        {"Model": IM["name"], "FP32 target acc": IM["fp32_acc"], "INT8 target acc": IM["int8_acc"],
         "INT8 CI low": IM["ci_lo"], "INT8 CI high": IM["ci_hi"], "INT8 .tflite (KB)": IM["int8_kb"], "Total params": IM["params"]},
    ])
    print("\n" + cmp.round(4).to_string(index=False))
    print(f"\nINT8 file size ratio (MSUN / MicroBrainNet-DA): {int8_kb / IM['int8_kb']:.1f}x   (both are measured .tflite files)")
    cmp.to_csv(f"{TABLES_DIR}/msun_vs_microbrainnet_int8.csv", index=False)

Params                : 9,775,427
FP32 size             : 37.29 MB (params x 4 bytes)
INT8 size, old est.   : 9.32 MB (params x 1 byte, Part 7)
INT8 .tflite MEASURED : 9990.3 KB = 9.76 MB   (FP32 .tflite 38149.7 KB, compression 3.82x)
measured / estimate   : 1.05x

LaTeX row (INT8 accuracy and INT8 size are both MEASURED on the .tflite file):

MSUN~\cite{wu2023msun} & ResNet-50 + Inception adaptation & LMMD + entropy min. & 73.42~[63.29, 82.28]$^{\dagger}$ & 9.76 MB \\
method                    : MSUN (Wu et al., 2023), this reproduction, real INT8 TFLite
seeds                     : [42]
params                    : 9775427
size_fp32_mb              : 37.29
int8_tflite_kb_measured   : 9990.3
tgt_test_acc_fp32         : 0.7722
tgt_test_acc_int8         : 0.7342
tgt_ci_int8               : (0.6329, 0.8228)
src_test_acc_fp32         : 0.8989
src_test_acc_int8         : 0.8191
train_peak_mem_mb         : 2807.6

                                Model  FP32 target acc  INT8 target acc  INT8 C

## Caveats for the INT8 part

1. **Same quantization method as the other notebooks, so the numbers are comparable.** Native TF/Keras rebuild -> `tf.lite.TFLiteConverter`, full-integer post-training quantization
   (per-channel int8 conv weights, per-tensor int8 activations, int8 input/output, `TFLITE_BUILTINS_INT8`), calibration on source-val + target-val images. No ONNX.
2. **PTQ, not QAT.** A small FP32 -> INT8 drop is possible; it is printed together with the INT8-vs-FP32 prediction agreement, and per-image predictions are saved in `tables/msun_tgt_test_predictions_int8_seed*.csv` for a paired comparison.
3. **What is quantized.** The inference form of MSUN: ResNet-50 up to `layer3` + the four substructures + pooling + FC. The LMMD and entropy terms exist only in training and are not part of the exported graph.
4. **Size is now a measured `.tflite` file.** The Part 7 `params x 1 byte` value is kept only as the "old estimate" for reference; the measured file also contains scales / zero-points, int32 biases and graph metadata.
5. **Size is not a like-for-like budget comparison.** MSUN keeps a ResNet-50 trunk (about 10 M parameters), so its INT8 file is tens of times larger than MicroBrainNet-DA's 245.6 KB. Compare accuracy per byte, not "fits the same MCU".
6. **Single seed (42).** There is no std. With 79 target-test images, 1 image = 1.27 accuracy points, so an FP32 -> INT8 change of one or two images is within noise; use the saved per-image predictions for a paired comparison.
7. **Calibration never touches `tgt_test`.** It uses 400 source-val and all target-val images (labels unused), the same recipe as the MicroBrainNet-DA export.
8. **Footnote $^{\dagger}$ still applies:** single-seed number with a bootstrap CI; do not place it next to a multi-seed mean +/- std without the footnote.